# Notebook 02 — NumPy, pandas, dan EDA

**Materi persiapan Ekshibisi Kecerdasan Artifisial · LKS Dikmen Nasional 2026**
Sasaran: siswa SMKN Bali Mandara, Program Keahlian Teknik Komputer dan Jaringan (TKJ)

Prasyarat: **Notebook 01** sudah dikerjakan sampai selesai.

---

### Apa yang akan kamu bisa setelah notebook ini

1. Memakai NumPy untuk berhitung atas banyak angka sekaligus, tanpa perulangan.
2. Memuat dataset publik langsung dari internet ke dalam sebuah tabel pandas.
3. Menjawab delapan pertanyaan wajib tentang sebuah dataset sebelum melatih model.
4. Menemukan data hilang, data ganda, dan pencilan (*outlier*) — lalu memutuskan
   apa yang dilakukan terhadapnya.
5. Membuat lima jenis grafik yang paling sering dibutuhkan, dan tahu kapan
   masing-masing dipakai.

### Istilah

**EDA** = *exploratory data analysis*, analisis data eksploratif. Tahap memandangi
data sebelum melatih model. Dilewati = model yang salah tanpa kamu sadari.

### Estimasi waktu

90–120 menit.

---

## Bagian 0 — Persiapan

`numpy`, `pandas`, `matplotlib`, dan `seaborn` sudah terpasang di Colab. Tidak
perlu `pip install` apa pun.

Nama pendek `np`, `pd`, `plt`, `sns` adalah kebiasaan yang dipakai hampir semua
orang. Ikuti saja — kode contoh di internet menganggapnya sudah ada.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Supaya angka desimal tidak ditampilkan dengan notasi ilmiah yang sulit dibaca
pd.set_option("display.float_format", lambda v: f"{v:,.3f}")
pd.set_option("display.width", 120)
sns.set_theme(style="whitegrid")

print("numpy      :", np.__version__)
print("pandas     :", pd.__version__)
print("seaborn    :", sns.__version__)

---

## Bagian 1 — NumPy: berhitung atas banyak angka sekaligus

Di Notebook 01 kita menjumlahkan nilai dengan `for`. NumPy melakukannya tanpa
`for`: satu operasi dikenakan ke seluruh isi array. Ini disebut **vektorisasi**.

Dua alasan memakainya: kodenya lebih pendek, dan jalannya jauh lebih cepat
(perhitungannya berlangsung di kode C, bukan di Python).

In [ ]:
# Cara lama (Notebook 01)
latensi_list = [12, 45, 150, 320, 60, 28]
naik_10_persen_lama = [x * 1.1 for x in latensi_list]

# Cara NumPy
latensi = np.array(latensi_list)
naik_10_persen = latensi * 1.1        # satu operasi untuk seluruh isi

print("list  :", naik_10_persen_lama)
print("numpy :", naik_10_persen)
print()
print("tipe  :", type(latensi), "| dtype:", latensi.dtype, "| bentuk:", latensi.shape)

In [ ]:
# Statistik dasar — semuanya satu baris
print("jumlah data :", latensi.size)
print("total       :", latensi.sum())
print("rata-rata   :", latensi.mean())
print("median      :", np.median(latensi))
print("simpangan   :", latensi.std())     # standar deviasi: seberapa menyebar
print("terkecil    :", latensi.min())
print("terbesar    :", latensi.max())

### Rata-rata dan median: selalu lihat keduanya

Rata-rata mudah tertarik oleh satu nilai ekstrem. Median (nilai tengah) tidak.
Kalau rata-rata jauh lebih besar daripada median, itu petunjuk ada pencilan di
sisi atas. Kebiasaan membandingkan keduanya akan sering menyelamatkanmu.

In [ ]:
tanpa_pencilan = np.array([12, 45, 60, 28, 33])
dengan_pencilan = np.array([12, 45, 60, 28, 33, 5000])   # satu nilai ekstrem

for nama, arr in [("tanpa pencilan", tanpa_pencilan), ("dengan pencilan", dengan_pencilan)]:
    print(f"{nama:<16} rata-rata {arr.mean():>8.1f} | median {np.median(arr):>6.1f}")

### Penyaringan dengan syarat (*boolean mask*)

Membandingkan array dengan sebuah angka menghasilkan array berisi `True`/`False`.
Array itu bisa langsung dipakai untuk mengambil isi yang memenuhi syarat. Pola
ini dipakai terus-menerus di pandas, jadi pahami di sini dulu.

In [ ]:
print("latensi        :", latensi)
print("latensi > 100  :", latensi > 100)             # array True/False
print("isinya         :", latensi[latensi > 100])    # hanya yang True
print("ada berapa     :", (latensi > 100).sum())     # True dihitung sebagai 1

# np.where: "kalau syarat benar pakai A, kalau tidak pakai B"
label = np.where(latensi < 100, "oke", "perlu dicek")
print()
for l, s in zip(latensi, label):
    print(f"{l:>4} ms -> {s}")

### Angka acak dan `seed`

`seed` membuat angka "acak" selalu sama setiap kali dijalankan. Wajib dipakai
supaya hasil pekerjaanmu bisa **diulang** orang lain — termasuk oleh dewan juri
yang menanyakan "kok angkanya beda dengan di laporan?".

In [ ]:
rng = np.random.default_rng(seed=42)      # 42 hanya angka pilihan, bebas
print("dengan seed 42:", rng.integers(0, 100, size=5))

rng = np.random.default_rng(seed=42)      # dibuat ulang dengan seed yang sama
print("diulang       :", rng.integers(0, 100, size=5))   # hasilnya identik

**Latihan 1.1.** Dari array `latensi`, hitung berapa persen ruang yang
latensinya di bawah 100 ms.

**Latihan 1.2.** Buat array 20 angka acak antara 1 dan 50 dengan `seed=7`, lalu
cetak rata-rata, median, dan jumlah nilai yang lebih besar dari 25.

---

## Bagian 2 — pandas: `Series` dan `DataFrame`

- **`Series`** = satu kolom, punya indeks.
- **`DataFrame`** = tabel, kumpulan `Series` yang berbagi indeks yang sama.

Kalau NumPy adalah array angka, pandas adalah tabel bernama kolom — seperti
lembar kerja Excel, tapi dikendalikan dengan kode.

In [ ]:
# Series
s = pd.Series([12, 45, 150, 320], index=["Lab 1", "Lab 2", "Perpus", "Aula"], name="latensi_ms")
print(s)
print()
print("Ambil satu nilai:", s["Perpus"])
print("Yang di atas 100:")
print(s[s > 100])

In [ ]:
# DataFrame dari dict — perhatikan bentuknya: nama kolom -> daftar nilai
df_jaringan = pd.DataFrame({
    "ruang": ["Lab 1", "Lab 2", "Perpus", "Aula", "R. Guru"],
    "latensi_ms": [12, 45, 150, 320, 60],
    "paket_hilang": [0.0, 0.4, 2.1, 7.5, 0.2],
    "jenis": ["lab", "lab", "umum", "umum", "kantor"],
})

df_jaringan

---

## Bagian 3 — Memuat dataset publik

Mulai di sini kita memakai data nyata, bukan buatan sendiri.

### Dataset: Palmer Penguins

| Keterangan | Isi |
|---|---|
| Nama | Palmer Archipelago (Antarctica) penguin data |
| Sumber asli | Gorman KB, Williams TD, Fraser WR (2014), *PLoS ONE* 9(3):e90081 |
| Pengumpul data | Dr. Kristen Gorman, Palmer Station LTER, Antarktika |
| Paket asal | `palmerpenguins` (Horst, Hill & Gorman, 2020) |
| Lisensi | CC0 — domain publik, bebas dipakai |
| Tautan yang kita pakai | `https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv` |
| Ukuran | 344 baris, 7 kolom |

**Mengapa dataset pinguin, bukan dataset sampah atau iklim?** Tiga alasan
teknis: ukurannya kecil sehingga cepat dijalankan di Colab gratis, isinya campur
angka dan kategori sehingga semua teknik yang perlu kamu pelajari muncul, dan
**ada data hilang di dalamnya** — yang justru penting untuk dilatih. Dataset yang
sudah bersih sempurna tidak mengajari apa pun tentang data nyata.

Pola kode di notebook ini sama persis untuk dataset apa pun. Di akhir Notebook 03
dan 04 ada latihan memakai data konsumsi listrik sekolah, yang temanya sejalan
dengan **Studi Kasus 2 (Lingkungan)** pada panduan LKS 2026.

### Dokumentasikan sumbernya — ini dinilai

Panduan LKS 2026 mewajibkan *disclosure*: seluruh sumber data, model, dan tools
harus dicantumkan. Biasakan menulis tabel seperti di atas sejak sekarang, bukan
sehari sebelum pengumpulan.

In [ ]:
URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/penguins.csv"

# read_csv bisa langsung menerima tautan internet, tidak harus berkas lokal
df = pd.read_csv(URL)

print("Dataset dimuat:", df.shape[0], "baris,", df.shape[1], "kolom")
df.head()

`.head()` menampilkan 5 baris pertama; `.head(10)` sepuluh. `.tail()` untuk baris
terakhir, `.sample(5)` untuk 5 baris acak.

Selalu lihat `.sample()` juga, bukan hanya `.head()`. Banyak dataset tersusun
berurutan, sehingga 5 baris pertama tidak mewakili keseluruhan.

In [ ]:
df.sample(5, random_state=42)     # random_state = seed, supaya bisa diulang

> **Kalau sel di atas galat** (`URLError`, `HTTPError`, atau macet lama):
> berarti Colab tidak bisa menjangkau internet. Alternatifnya:
> jalankan `df = sns.load_dataset("penguins")` — seaborn punya salinan dataset
> yang sama. Kalau masih gagal, unduh CSV-nya dari tautan di atas memakai
> peramban, lalu unggah ke Colab dengan `files.upload()` (lihat Notebook 01
> Bagian 8) dan baca dengan `pd.read_csv("penguins.csv")`.

---

## Bagian 4 — Delapan pertanyaan wajib sebelum melatih model

Ini daftar periksa yang sama untuk setiap dataset. Jawab semuanya, tulis
jawabannya, lalu baru lanjut.

| # | Pertanyaan | Alatnya |
|---|---|---|
| 1 | Berapa banyak baris dan kolom? | `df.shape` |
| 2 | Apa arti setiap kolom, dan satuannya? | dokumentasi dataset |
| 3 | Apa tipe data setiap kolom? Sudah benar? | `df.info()` |
| 4 | Ada nilai hilang? Di kolom mana, berapa banyak? | `df.isna().sum()` |
| 5 | Ada baris ganda? | `df.duplicated().sum()` |
| 6 | Bagaimana sebaran setiap kolom angka? | `df.describe()`, histogram |
| 7 | Berapa banyak tiap kategori? Seimbang? | `df["kolom"].value_counts()` |
| 8 | Kolom mana yang akan menjadi **target**? | keputusanmu |

In [ ]:
# 1 & 3 — bentuk, nama kolom, tipe data, dan jumlah nilai terisi
df.info()

Cara membaca keluaran `.info()`:

- `RangeIndex: 344 entries` → 344 baris.
- Kolom `Non-Null Count` menunjukkan berapa nilai yang **terisi**. Kalau
  angkanya kurang dari 344, berarti ada nilai hilang di kolom itu.
- `Dtype`: `float64`/`int64` = angka, `object` (atau `str` pada pandas versi
  baru) = teks.

**Waspadai `object`/`str` pada kolom yang seharusnya angka.** Itu tanda ada nilai
seperti `"tidak terbaca"` atau `"-"` di dalamnya — persis masalah yang kita
tangani di Notebook 01 Bagian 9.

### 2 — Arti setiap kolom

| Kolom | Arti | Satuan |
|---|---|---|
| `species` | jenis pinguin: Adelie, Chinstrap, Gentoo | kategori |
| `island` | pulau tempat pengamatan | kategori |
| `bill_length_mm` | panjang paruh | milimeter |
| `bill_depth_mm` | ketebalan paruh | milimeter |
| `flipper_length_mm` | panjang sayap (*flipper*) | milimeter |
| `body_mass_g` | massa tubuh | gram |
| `sex` | jenis kelamin | kategori |

Tanpa tabel ini, model yang kamu latih hanya angka tanpa makna — dan kamu tidak
akan bisa menjawab pertanyaan juri "apa arti fitur yang paling berpengaruh itu?".

In [ ]:
# 6 — ringkasan statistik kolom angka
df.describe()

Cara membacanya per baris:

- `count` — jumlah nilai terisi (bukan jumlah baris!)
- `mean`, `std` — rata-rata dan sebarannya
- `min`, `max` — nilai terkecil dan terbesar; di sini tempat pencilan atau data
  mustahil (umur `-5`, harga `0`) biasanya terlihat
- `25%`, `50%`, `75%` — kuartil; `50%` adalah median

Bandingkan `mean` dan `50%` setiap kolom. Selisih besar = sebaran tidak simetris.

In [ ]:
# describe untuk kolom teks: berapa kategori, mana yang terbanyak
df[["species", "island", "sex"]].describe()

In [ ]:
# 4 — nilai hilang
hilang = df.isna().sum()
print("Jumlah nilai hilang per kolom:")
print(hilang[hilang > 0])
print()
print(f"Baris yang punya minimal satu nilai hilang: {df.isna().any(axis=1).sum()} dari {len(df)}")
print()

# 5 — baris ganda
print("Baris ganda:", df.duplicated().sum())

Dua baris yang punya nilai hilang di hampir semua kolom pengukuran. Satu yang
lain hanya kehilangan `sex`. Cara menanganinya dibahas di Bagian 7 — tapi
sebelum itu, **lihat dulu barisnya**. Jangan menghapus apa pun yang belum kamu
lihat.

In [ ]:
# Lihat baris yang bermasalah sebelum memutuskan apa pun
df[df.isna().any(axis=1)]

In [ ]:
# 7 — jumlah tiap kategori
for kolom in ["species", "island", "sex"]:
    print(f"--- {kolom} ---")
    print(df[kolom].value_counts(dropna=False))    # dropna=False supaya NaN ikut terhitung
    print()

`value_counts(normalize=True)` memberi proporsinya, bukan hitungannya. Ini yang
dipakai untuk menilai **keseimbangan kelas** — apakah satu kategori jauh lebih
banyak daripada yang lain.

Kalau sangat tidak seimbang (misalnya 95% : 5%), akurasi menjadi angka yang
menipu: model yang selalu menjawab kelas mayoritas sudah "akurat 95%" tanpa
belajar apa pun. Ini dibahas lagi di Notebook 03.

In [ ]:
print(df["species"].value_counts(normalize=True).mul(100).round(1))
print()
print("Kelas terbesar :", df["species"].value_counts(normalize=True).max().round(3))
print("-> model yang selalu menjawab kelas itu akan 'akurat' sebesar angka tersebut.")

Angka terakhir itu disebut **baseline mayoritas**. Model apa pun yang kamu buat
harus mengalahkannya. Kalau tidak, model itu tidak berguna, seberapa pun canggih
algoritmanya. Catat angkanya sekarang.

**Latihan 4.1.** Jawab kedelapan pertanyaan di atas untuk dataset ini dalam sel
teks baru, dengan bahasamu sendiri. Ini latihan terpenting di notebook ini.

---

## Bagian 5 — Memilih baris dan kolom

Tiga cara, dan kapan memakai yang mana:

| Cara | Untuk |
|---|---|
| `df["kolom"]` | mengambil satu kolom |
| `df.loc[baris, kolom]` | memilih dengan **nama** atau dengan syarat |
| `df.iloc[baris, kolom]` | memilih dengan **nomor** urut |

In [ ]:
print(type(df["species"]))                 # satu kolom -> Series
print(type(df[["species", "body_mass_g"]]))  # dua kurung siku -> DataFrame

df[["species", "island", "body_mass_g"]].head()

In [ ]:
# iloc — dengan nomor. Pola [baris, kolom]
print("Baris ke-0, kolom ke-0 :", df.iloc[0, 0])
print()
print("Tiga baris pertama, dua kolom pertama:")
print(df.iloc[0:3, 0:2])

In [ ]:
# loc dengan syarat — inilah pola yang paling sering dipakai
berat = df.loc[df["body_mass_g"] > 5500, ["species", "island", "body_mass_g"]]
print(f"Ada {len(berat)} pinguin dengan massa di atas 5500 g:")
berat.head()

Menggabung beberapa syarat: pakai `&` untuk "dan", `|` untuk "atau". **Setiap
syarat wajib dibungkus tanda kurung** — tanpa kurung, Python salah menafsirkan
urutan operasinya dan memunculkan galat yang membingungkan.

In [ ]:
pilihan = df.loc[
    (df["species"] == "Gentoo") & (df["body_mass_g"] > 5500),
    ["species", "sex", "body_mass_g", "flipper_length_mm"],
]
print(f"Gentoo dengan massa > 5500 g: {len(pilihan)} baris")
pilihan.head()

In [ ]:
# isin untuk beberapa nilai sekaligus, ~ untuk membalik syarat
dua_jenis = df[df["species"].isin(["Adelie", "Gentoo"])]
bukan_dream = df[~df["island"].isin(["Dream"])]

print("Adelie + Gentoo     :", len(dua_jenis), "baris")
print("Bukan pulau Dream   :", len(bukan_dream), "baris")

# sort_values untuk mengurutkan
df.sort_values("body_mass_g", ascending=False).head(3)

**Latihan 5.1.** Ambil semua pinguin `Chinstrap` berjenis kelamin `FEMALE`, lalu
tampilkan rata-rata `bill_length_mm`-nya.

---

## Bagian 6 — Kolom baru, `groupby`, dan tabel ringkasan

Membuat kolom baru dari kolom yang ada disebut **rekayasa fitur** (*feature
engineering*). Ini salah satu cara paling murah untuk memperbaiki model — sering
lebih berpengaruh daripada mengganti algoritma.

In [ ]:
# Kolom baru dari perhitungan antar kolom
df["massa_kg"] = df["body_mass_g"] / 1000
df["rasio_paruh"] = df["bill_length_mm"] / df["bill_depth_mm"]

# Kolom baru dari kategori: pd.cut memotong angka menjadi kelompok
df["kelas_massa"] = pd.cut(
    df["body_mass_g"],
    bins=[0, 3500, 4500, 5500, 10000],
    labels=["ringan", "sedang", "berat", "sangat berat"],
)

df[["species", "body_mass_g", "massa_kg", "rasio_paruh", "kelas_massa"]].head()

`groupby` adalah versi pandas dari pekerjaan 8 baris yang kita tulis dengan
`dict` di Notebook 01 Bagian 8. Polanya: **kelompokkan → hitung**.

In [ ]:
# Satu kolom, satu perhitungan
print(df.groupby("species")["body_mass_g"].mean().round(1))
print()

# Satu kolom, beberapa perhitungan
print(df.groupby("species")["body_mass_g"].agg(["count", "mean", "std", "min", "max"]).round(1))

In [ ]:
# Beberapa kolom sekaligus, dengan nama hasil yang kita tentukan sendiri
ringkasan = df.groupby("species").agg(
    jumlah=("species", "size"),
    massa_rata=("body_mass_g", "mean"),
    sayap_rata=("flipper_length_mm", "mean"),
    paruh_rata=("bill_length_mm", "mean"),
).round(1)

ringkasan

Tabel di atas sudah cukup untuk melihat satu hal penting: **Gentoo jauh lebih
berat dan bersayap lebih panjang.** Artinya kedua kolom itu kemungkinan besar
berguna untuk membedakan spesies — dugaan yang akan kita uji di Notebook 03.

EDA memang bekerja seperti ini: melihat angka, menduga, lalu menguji.

In [ ]:
# Dua pengelompokan sekaligus
print(df.groupby(["species", "sex"])["body_mass_g"].mean().round(1))
print()

# crosstab: menghitung kombinasi dua kolom kategori
print(pd.crosstab(df["species"], df["island"]))

Perhatikan tabel terakhir: **Gentoo hanya ada di Biscoe, Chinstrap hanya di
Dream.** Itu informasi yang sangat kuat — dan juga sebuah peringatan. Model yang
belajar dari data ini bisa saja hanya menghafal "pulau Dream berarti Chinstrap",
yang tidak berlaku di luar wilayah pengamatan ini.

Ini contoh nyata dari apa yang disebut **kebocoran informasi geografis**, dan
bagian dari pembahasan Responsible AI di Notebook 03. Temuan seperti ini hanya
muncul kalau kamu benar-benar melakukan EDA.

**Latihan 6.1.** Buat tabel ringkasan rata-rata `flipper_length_mm` per
kombinasi `island` dan `sex`.

---

## Bagian 7 — Menangani nilai hilang

Tiga pilihan, masing-masing punya konsekuensi. Tidak ada yang selalu benar.

| Pilihan | Kode | Kapan pantas |
|---|---|---|
| Hapus barisnya | `df.dropna()` | jumlah barisnya sedikit, dan hilangnya acak |
| Hapus kolomnya | `df.drop(columns=[...])` | sebagian besar isi kolom itu hilang |
| Isi dengan perkiraan | `fillna()`, `SimpleImputer` | data sedikit, tiap baris berharga |

Aturan praktis: **sebutkan berapa baris yang kamu buang, dan mengapa.** Menghapus
data tanpa menghitungnya adalah cara termudah untuk menipu diri sendiri.

> Peringatan yang akan diulang di Notebook 03: kalau nilai hilang diisi dengan
> rata-rata, **rata-rata itu harus dihitung dari data latih saja**, bukan dari
> seluruh dataset. Kalau dari seluruh dataset, informasi data uji ikut bocor ke
> dalam pelatihan dan hasil pengujiannya jadi terlalu bagus. Cara amannya adalah
> `Pipeline`, yang dipakai mulai Notebook 03.

In [ ]:
sebelum = len(df)

# Pilihan 1 — hapus baris yang kolom pengukurannya hilang
kolom_ukur = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
df_bersih = df.dropna(subset=kolom_ukur).copy()

print(f"Sebelum      : {sebelum} baris")
print(f"Sesudah      : {len(df_bersih)} baris")
print(f"Dibuang      : {sebelum - len(df_bersih)} baris "
      f"({(sebelum - len(df_bersih)) / sebelum * 100:.1f}%)")
print()
print("Sisa nilai hilang:")
print(df_bersih.isna().sum()[lambda s: s > 0])

In [ ]:
# Pilihan 3 — kolom sex masih punya nilai hilang. Isi dengan penanda eksplisit.
# "Tidak diketahui" lebih jujur daripada menebak jenis kelaminnya.
df_bersih["sex"] = df_bersih["sex"].fillna("TIDAK DIKETAHUI")

print(df_bersih["sex"].value_counts())
print()
print("Total nilai hilang sekarang:", int(df_bersih.isna().sum().sum()))

Catat keputusannya supaya bisa dilaporkan:

> Dari 344 baris, 2 baris dibuang karena seluruh kolom pengukurannya kosong
> (0,6%). Kolom `sex` yang kosong pada 9 baris diisi dengan kategori
> `TIDAK DIKETAHUI` alih-alih ditebak. Tersisa 342 baris untuk pemodelan.

Kalimat seperti itu — dengan angkanya — adalah yang dicari juri pada bagian
dokumentasi. Bukan "data sudah dibersihkan".

---

## Bagian 8 — Visualisasi

Lima grafik yang menjawab lima pertanyaan berbeda. Pilih berdasarkan
pertanyaannya, bukan berdasarkan yang paling bagus dilihat.

| Pertanyaan | Grafik |
|---|---|
| Bagaimana sebaran satu kolom angka? | histogram |
| Apakah sebarannya berbeda antar kelompok? | boxplot |
| Apakah dua kolom angka berhubungan? | scatter plot |
| Berapa banyak tiap kategori? | bar chart |
| Kolom mana yang saling berkaitan? | heatmap korelasi |

In [ ]:
# 1 — HISTOGRAM: sebaran satu kolom angka
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].hist(df_bersih["body_mass_g"], bins=25, color="#4C72B0", edgecolor="white")
ax[0].set_title("Sebaran massa tubuh")
ax[0].set_xlabel("massa (gram)")
ax[0].set_ylabel("jumlah pinguin")

# Dipisah per spesies: terlihat bahwa yang di atas adalah dua kelompok, bukan satu
sns.histplot(data=df_bersih, x="body_mass_g", hue="species", bins=25,
             multiple="stack", ax=ax[1])
ax[1].set_title("Massa tubuh per spesies")
ax[1].set_xlabel("massa (gram)")

plt.tight_layout()
plt.show()

Bandingkan kedua grafik. Grafik kiri menunjukkan sebaran bergelombang dua puncak
dan mudah disalahartikan. Grafik kanan menjelaskan sebabnya: Gentoo memang
kelompok tersendiri yang lebih berat.

**Pelajaran umum:** sebaran yang aneh sering berarti data itu sebetulnya
gabungan beberapa kelompok. Pisahkan dulu sebelum menyimpulkan.

In [ ]:
# 2 — BOXPLOT: membandingkan sebaran antar kelompok
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

sns.boxplot(data=df_bersih, x="species", y="flipper_length_mm", hue="species",
            legend=False, ax=ax[0])
ax[0].set_title("Panjang sayap per spesies")

sns.boxplot(data=df_bersih, x="island", y="body_mass_g", hue="island",
            legend=False, ax=ax[1])
ax[1].set_title("Massa tubuh per pulau")

plt.tight_layout()
plt.show()

Cara membaca boxplot: kotak berisi 50% data bagian tengah, garis di dalamnya
adalah median, kumis memanjang sampai data yang masih dianggap wajar, dan titik
di luar kumis adalah **calon pencilan**.

"Calon", bukan "pencilan". Titik di luar kumis belum tentu salah — bisa jadi
memang ada pinguin yang sangat besar. Diperiksa dulu, jangan langsung dibuang.

In [ ]:
# 3 — SCATTER PLOT: hubungan dua kolom angka
fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))

ax[0].scatter(df_bersih["flipper_length_mm"], df_bersih["body_mass_g"],
              alpha=0.6, color="#55A868")
ax[0].set_xlabel("panjang sayap (mm)")
ax[0].set_ylabel("massa tubuh (g)")
ax[0].set_title("Tanpa pembeda warna")

sns.scatterplot(data=df_bersih, x="flipper_length_mm", y="body_mass_g",
                hue="species", style="species", s=55, ax=ax[1])
ax[1].set_title("Diwarnai per spesies")
ax[1].legend(title="spesies", loc="upper left", fontsize=8)

plt.tight_layout()
plt.show()

Grafik kanan adalah inti dari seluruh Notebook 03. Ketiga spesies membentuk
kelompok yang **hampir tidak bertumpuk** pada dua kolom ini saja.

Artinya: masalah ini bisa dipecahkan model klasifikasi sederhana, dan kita sudah
tahu jawabannya **sebelum** melatih apa pun. Kalau ternyata akurasinya rendah,
berarti ada kesalahan pada kodenya — bukan pada datanya. Kemampuan menduga hasil
lebih dulu ini yang membedakan orang yang paham dari orang yang menyalin kode.

In [ ]:
# 4 — BAR CHART: jumlah tiap kategori
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

df_bersih["species"].value_counts().plot(kind="bar", ax=ax[0], color="#C44E52")
ax[0].set_title("Jumlah data per spesies")
ax[0].set_ylabel("jumlah baris")
ax[0].tick_params(axis="x", rotation=0)

pd.crosstab(df_bersih["island"], df_bersih["species"]).plot(kind="bar", stacked=True, ax=ax[1])
ax[1].set_title("Spesies per pulau")
ax[1].set_ylabel("jumlah baris")
ax[1].tick_params(axis="x", rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
# 5 — HEATMAP KORELASI: kolom angka mana yang saling berkaitan
kolom_angka = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
korelasi = df_bersih[kolom_angka].corr()

plt.figure(figsize=(6.5, 5))
sns.heatmap(korelasi, annot=True, fmt=".2f", cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5)
plt.title("Korelasi antar kolom pengukuran")
plt.tight_layout()
plt.show()

print(korelasi.round(2))

**Korelasi** berkisar dari -1 sampai 1:

- mendekati **+1** — naik bersamaan
- mendekati **0** — tidak ada hubungan garis lurus
- mendekati **-1** — yang satu naik, yang lain turun

Di sini `flipper_length_mm` dan `body_mass_g` berkorelasi kuat (sekitar 0,87):
pinguin bersayap panjang memang lebih berat. Wajar.

Dua hal yang harus kamu ingat tentang korelasi:

1. **Korelasi bukan sebab-akibat.** Sayap panjang tidak *menyebabkan* berat
   bertambah; keduanya sama-sama akibat ukuran tubuh.
2. **Korelasi hanya mengukur hubungan garis lurus.** Hubungan berbentuk kurva
   bisa terbaca 0. Karena itu scatter plot tetap perlu dilihat.

In [ ]:
# pairplot: semua scatter + histogram sekaligus. Berat, tapi sekali jalan
# memberi gambaran paling lengkap. Jangan dipakai untuk dataset dengan
# puluhan kolom (grafiknya menjadi tidak terbaca).
sns.pairplot(df_bersih[kolom_angka + ["species"]], hue="species",
             diag_kind="hist", height=1.9, plot_kws={"s": 22, "alpha": 0.7})
plt.suptitle("Pairplot: semua kombinasi kolom pengukuran", y=1.02)
plt.show()

**Latihan 8.1.** Buat boxplot `bill_length_mm` per `species`. Spesies mana yang
paruhnya paling panjang?

**Latihan 8.2.** Buat scatter plot `bill_length_mm` (sumbu x) terhadap
`bill_depth_mm` (sumbu y), diwarnai per `species`. Apakah ketiga kelompoknya
terpisah sebaik grafik di Bagian 8 nomor 3?

---

## Bagian 9 — Cek pemahaman

Lengkapi sel di bawah sampai semua pemeriksaan lolos.

In [ ]:
# ---- Kerjakan: ganti None dengan kode yang benar ----

# a) Berapa baris pada df_bersih?
jumlah_baris = None

# b) Rata-rata body_mass_g untuk spesies Gentoo saja (bulatkan ke 1 desimal)
massa_gentoo = None

# c) Berapa banyak pinguin yang flipper_length_mm-nya lebih dari 210 mm?
sayap_panjang = None

# d) Nama spesies dengan jumlah data paling sedikit
spesies_tersedikit = None


# ---- Pemeriksaan otomatis ----
assert jumlah_baris == 342, f"(a) seharusnya 342, dapatnya {jumlah_baris}"
assert massa_gentoo is not None and abs(massa_gentoo - 5076.0) < 1.0, \
    f"(b) seharusnya sekitar 5076.0, dapatnya {massa_gentoo}"
assert sayap_panjang == 100, f"(c) seharusnya 100, dapatnya {sayap_panjang}"
assert spesies_tersedikit == "Chinstrap", f"(d) seharusnya 'Chinstrap', dapatnya {spesies_tersedikit}"

print("Semua pemeriksaan lolos. Lanjut ke Notebook 03.")

---

## Ringkasan

### Yang paling sering dipakai

```python
df = pd.read_csv(URL)          # muat data
df.shape, df.info()            # bentuk dan tipe data
df.head(), df.sample(5)        # lihat isinya
df.describe()                  # ringkasan statistik
df.isna().sum()                # nilai hilang per kolom
df.duplicated().sum()          # baris ganda
df["kolom"].value_counts()     # hitung per kategori
df[df["kolom"] > 10]           # saring baris
df.groupby("a")["b"].mean()    # kelompokkan lalu hitung
df.dropna(subset=[...])        # buang baris tak lengkap
df["baru"] = df["a"] / df["b"] # kolom baru
df.corr(numeric_only=True)     # korelasi
```

### Temuan EDA kita atas dataset pinguin

1. 344 baris, 7 kolom. Dua baris hampir kosong, sembilan baris kehilangan `sex`.
   Setelah dibersihkan: 342 baris.
2. Tidak ada baris ganda.
3. Tiga spesies dengan jumlah tidak merata; kelas terbesar sekitar 44%
   — itulah baseline yang harus dikalahkan.
4. `flipper_length_mm` dan `body_mass_g` memisahkan Gentoo dengan sangat jelas.
5. Spesies terkait erat dengan pulau — berguna untuk model, tapi berisiko
   membuat model hanya menghafal lokasi.

### Berikutnya

**Notebook 03 — Studi Kasus Klasifikasi.** Dugaan nomor 4 di atas akan kita uji
dengan model sungguhan, lengkap dengan pembagian data latih/uji, baseline,
perbandingan beberapa algoritma, dan pembacaan *confusion matrix*.